# Advanced — Multi-Input + Multi-Label Outputs

Optimize a signature that fuses two text inputs and predicts four outputs, including a list-valued multi-label field.

**Dataset:** `uspto_patents.json` — 100 historic US patents with title + abstract as inputs and four structured outputs: primary category, subcategory tags (multi-label), inventor count, decade. (USPTO records, US-Government public domain + factual.)

**What you'll learn:**
1. Multi-input signatures with text fusion
2. List-valued (multi-label) output fields
3. A composite metric that combines categorical accuracy + Jaccard similarity
4. Inspect per-example errors with `/test-results`

**Prerequisites:** Complete [01_quickstart.ipynb](01_quickstart.ipynb) and [03_creative_tasks.ipynb](03_creative_tasks.ipynb).

In [ ]:
import json
import os
from pathlib import Path

import dspy

from skynet_client import DSPyServiceClient, JobMonitor, serialize_source

In [ ]:
BASE_URL = os.getenv("DSPY_SERVICE_URL", "http://localhost:8000")
LM_BASE_URL = os.getenv("DSPY_LM_BASE_URL", "https://api.openai.com/v1")

# LM_API_KEY is the canonical name; OPENAI_API_KEY is accepted as a
# fallback for backwards-compat and the public OpenAI dev path.
LM_API_KEY = os.getenv("LM_API_KEY") or os.getenv("OPENAI_API_KEY")
if not LM_API_KEY:
    raise ValueError("Set LM_API_KEY (or OPENAI_API_KEY) — any non-empty token works for self-hosted gateways.")

MODEL_CONFIG = {
    "name": "openai/gpt-4o-mini",
    "base_url": LM_BASE_URL,
    "model_type": "responses",
    "temperature": 1.0,
    "max_tokens": 20000,
    "extra": {"api_key": LM_API_KEY},
}

dspy.configure(
    lm=dspy.LM(
        "openai/gpt-4o-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=20000,
        api_base=LM_BASE_URL,
        api_key=LM_API_KEY,
    )
)

client = DSPyServiceClient(BASE_URL)
client.health()

## 1. Load Dataset

Each row has 2 inputs (title, abstract) and 4 outputs — note that `subcategories` is a list of strings.

In [ ]:
with open(Path("../../data/uspto_patents.json")) as f:
    dataset = json.load(f)

print(f"{len(dataset)} patents")
print(f"Fields: {list(dataset[0].keys())}")
print()
sample = dataset[7]
for k, v in sample.items():
    print(f"  {k}: {v}")

## 2. Multi-Input + Multi-Label Signature

`subcategories: list[str]` makes DSPy emit a JSON-style list output. The optimizer learns to map (title, abstract) into the right tag set.

In [ ]:
class PatentClassifier(dspy.Signature):
    """Classify a US patent into a primary category, subcategory tags, inventor count, and decade."""
    title: str = dspy.InputField(desc="Patent title as filed")
    abstract: str = dspy.InputField(desc="One-sentence description of the claimed invention")

    primary_category: str = dspy.OutputField(
        desc="One of: communication, transportation, computing, medical, energy, "
             "chemistry, household, entertainment, aerospace, manufacturing"
    )
    subcategories: list[str] = dspy.OutputField(
        desc="1-3 lowercase subcategory tags drawn from the closed taxonomy "
             "(e.g. telegraphy, telephony, semiconductor, polymer, lighting)"
    )
    inventor_count: int = dspy.OutputField(desc="Number of named inventors on the patent")
    decade: str = dspy.OutputField(desc="Decade granted as a 5-character string (e.g. \"1880s\")")


SIGNATURE_CODE = serialize_source(PatentClassifier)
print(SIGNATURE_CODE)

## 3. Composite Metric

Four outputs, three different scoring rules: exact match for the categoricals, Jaccard for the multi-label tags, and a numeric tolerance for inventor count. Average them into a single score and surface the worst offender in the feedback.

In [ ]:
METRIC_CODE = '''
def patent_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    def _norm_tag(t):
        return str(t).strip().lower()

    def _to_list(x):
        if isinstance(x, list):
            return [_norm_tag(t) for t in x if t]
        if isinstance(x, str):
            return [_norm_tag(t) for t in x.replace("[", "").replace("]", "").split(",") if t.strip()]
        return []

    cat_ok = str(gold.primary_category).strip().lower() == str(pred.primary_category).strip().lower()
    decade_ok = str(gold.decade).strip().lower() == str(pred.decade).strip().lower()

    g_tags = set(_to_list(gold.subcategories))
    p_tags = set(_to_list(pred.subcategories))
    if g_tags or p_tags:
        jaccard = len(g_tags & p_tags) / len(g_tags | p_tags)
    else:
        jaccard = 1.0

    try:
        ic_diff = abs(int(gold.inventor_count) - int(pred.inventor_count))
    except (TypeError, ValueError):
        ic_diff = 99
    ic_score = 1.0 if ic_diff == 0 else (0.5 if ic_diff == 1 else 0.0)

    score = round((int(cat_ok) + jaccard + ic_score + int(decade_ok)) / 4, 2)

    parts = []
    if not cat_ok:
        parts.append(f"primary_category: expected {gold.primary_category!r}, got {pred.primary_category!r}")
    if jaccard < 1.0:
        parts.append(f"subcategories: expected {sorted(g_tags)}, got {sorted(p_tags)} (Jaccard={jaccard:.2f})")
    if ic_score < 1.0:
        parts.append(f"inventor_count: expected {gold.inventor_count}, got {pred.inventor_count}")
    if not decade_ok:
        parts.append(f"decade: expected {gold.decade!r}, got {pred.decade!r}")

    if not parts:
        return dspy.Prediction(score=1.0, feedback="All four fields correct.")
    feedback = "; ".join(parts)
    return dspy.Prediction(score=score, feedback=feedback)
'''

## 4. Validate

In [ ]:
validation = client.validate_code({
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "column_mapping": {
        "inputs": {"title": "title", "abstract": "abstract"},
        "outputs": {
            "primary_category": "primary_category",
            "subcategories": "subcategories",
            "inventor_count": "inventor_count",
            "decade": "decade",
        },
    },
})

print(f"Valid: {validation['valid']}")
if validation.get("errors"):
    print(f"Errors: {validation['errors']}")
if validation.get("signature_fields"):
    print(f"Fields: {validation['signature_fields']}")

## 5. Submit Optimization

In [ ]:
payload = {
    "username": "notebook-user",
    "module_name": "dspy.ChainOfThought",
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "optimizer_name": "dspy.GEPA",
    "optimizer_kwargs": {"auto": "light", "num_threads": 8},
    "compile_kwargs": {},
    "dataset": dataset,
    "column_mapping": {
        "inputs": {"title": "title", "abstract": "abstract"},
        "outputs": {
            "primary_category": "primary_category",
            "subcategories": "subcategories",
            "inventor_count": "inventor_count",
            "decade": "decade",
        },
    },
    "split_fractions": {"train": 0.5, "val": 0.3, "test": 0.2},
    "shuffle": True,
    "seed": 42,
    "model_config": MODEL_CONFIG,
    "reflection_model_config": MODEL_CONFIG,
}

optimization_id = client.submit(payload)
print(f"Submitted: {optimization_id}")

In [ ]:
monitor = JobMonitor(client, optimization_id)
result = monitor.poll(interval=5)

if result["status"] == "success":
    r = result["result"]
    print(f"\nBaseline score:  {r.get('baseline_test_metric', 'N/A')}")
    print(f"Optimized score: {r.get('optimized_test_metric', 'N/A')}")
    print(f"Runtime: {r.get('runtime_seconds', 0):.0f}s")
else:
    print(f"\nFailed: {result.get('message')}")

## 6. Inspect Per-Example Failures

The `/test-results` endpoint returns scores per example, both before and after optimization — useful for spotting where the multi-label tagging is still falling short.

In [ ]:
test_results = client.test_results(optimization_id)
baseline = test_results.get("baseline", [])
optimized = test_results.get("optimized", [])

print(f"{len(baseline)} test examples\n")
for i, (b, o) in enumerate(zip(baseline[:8], optimized[:8], strict=True)):
    delta = (o.get("score") or 0) - (b.get("score") or 0)
    arrow = "↑" if delta > 0 else ("→" if delta == 0 else "↓")
    print(f"  ex {i}: baseline={b.get('score')}  optimized={o.get('score')}  {arrow}")

## 7. Local Inference

In [ ]:
program = client.load_program(optimization_id)

response = program(
    title="Improvement in incandescent lamps",
    abstract="An improved long-lived filament lamp using a treated bamboo filament for general illumination.",
)
print(f"  primary_category: {response.primary_category}")
print(f"  subcategories:    {response.subcategories}")
print(f"  inventor_count:   {response.inventor_count}")
print(f"  decade:           {response.decade}")

## What's Next

- **[05_gepa_showcase.ipynb](05_gepa_showcase.ipynb)** — Push GEPA on a hard constraint-reasoning task.
- **API Reference** — `http://localhost:8000/reference`